# Speaker Identification with GMMs

Extract MFCC features from speech, train a Gaussian Mixture Model per speaker, and classify unknown utterances.

Uses `v_melcepst` for feature extraction, `v_gaussmix` for GMM training, and `v_gaussmixp` for scoring. No scikit-learn or PyTorch needed for the core pipeline.

Dataset: [EmoDB](http://emodb.bilderbar.info/) — 10 speakers, 535 utterances.

In [ ]:
# Install dependencies if needed (e.g. on Colab)
# %pip install pyvoicebox scikit-learn

In [ ]:
import os
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from pyvoicebox import v_melcepst, v_gaussmix, v_gaussmixp
from sklearn.manifold import TSNE

## Download EmoDB

In [ ]:
data_dir = "emodb"
zip_path = "emodb.zip"
wav_dir = os.path.join(data_dir, "wav")

if not os.path.isdir(wav_dir):
    if not os.path.exists(zip_path):
        print("Downloading EmoDB (~26 MB)...")
        urllib.request.urlretrieve("http://emodb.bilderbar.info/download/download.zip", zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_dir)
    print("Done.")
else:
    print(f"EmoDB already extracted in {wav_dir}")

# Index all WAV files by speaker (first 2 characters of filename)
speaker_files = {}
for fname in sorted(os.listdir(wav_dir)):
    if not fname.endswith('.wav'):
        continue
    spk = fname[:2]
    speaker_files.setdefault(spk, []).append(os.path.join(wav_dir, fname))

speakers = sorted(speaker_files.keys())
print(f"{len(speakers)} speakers: {speakers}")
for spk in speakers:
    print(f"  {spk}: {len(speaker_files[spk])} clips")

In [ ]:
# Listen to one sample from each speaker
for spk in speakers[:5]:  # first 5 to keep it short
    sig, rate = sf.read(speaker_files[spk][0])
    print(f"Speaker {spk}:")
    display(Audio(sig, rate=rate))

## Extract MFCC features

For each clip, extract 12 MFCCs. Pool all frames per speaker for GMM training.
We use a 70/30 train/test split per speaker.

In [ ]:
def extract_mfcc(wav_path, n_cep=12):
    """Extract MFCCs from a WAV file."""
    signal, fs = sf.read(wav_path)
    if signal.ndim > 1:
        signal = signal.mean(axis=1)
    mfcc, _ = v_melcepst(signal, fs, 'M', n_cep)
    return mfcc

train_features = {}
test_clips = {}

for spk in speakers:
    files = speaker_files[spk]
    n_train = int(len(files) * 0.7)
    train_frames = []
    test_list = []
    for i, fpath in enumerate(files):
        mfcc = extract_mfcc(fpath)
        if i < n_train:
            train_frames.append(mfcc)
        else:
            test_list.append(mfcc)
    train_features[spk] = np.vstack(train_frames)
    test_clips[spk] = test_list

for spk in speakers:
    print(f"{spk}: {train_features[spk].shape[0]} train frames, "
          f"{len(test_clips[spk])} test clips")

## Train a GMM per speaker

`v_gaussmix` fits a diagonal-covariance GMM via EM. We use 8 mixture components per speaker.

In [ ]:
n_mix = 8
gmm_models = {}

for spk in speakers:
    data = train_features[spk]
    m, v, w, g, f, pp, gg = v_gaussmix(data, m0=n_mix)
    gmm_models[spk] = (m, v, w)
    print(f"{spk}: {n_mix} mixtures, avg log-prob = {g:.2f}")

## Classify test clips

For each test clip, score all its frames against each speaker's GMM using `v_gaussmixp`. The speaker with the highest total log-likelihood wins.

In [ ]:
correct = 0
total = 0
confusion = np.zeros((len(speakers), len(speakers)), dtype=int)

for true_idx, true_spk in enumerate(speakers):
    for clip_mfcc in test_clips[true_spk]:
        scores = []
        for spk in speakers:
            m, v, w = gmm_models[spk]
            lp, _, _, _ = v_gaussmixp(clip_mfcc, m, v, w)
            scores.append(np.sum(lp))
        
        pred_idx = np.argmax(scores)
        confusion[true_idx, pred_idx] += 1
        correct += (pred_idx == true_idx)
        total += 1

accuracy = correct / total
print(f"Accuracy: {correct}/{total} = {accuracy:.0%}")

## Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(confusion, cmap='Blues', interpolation='nearest')
ax.set_xticks(range(len(speakers)))
ax.set_yticks(range(len(speakers)))
ax.set_xticklabels(speakers, fontsize=9)
ax.set_yticklabels(speakers, fontsize=9)
ax.set_xlabel('Predicted speaker')
ax.set_ylabel('True speaker')
ax.set_title(f'Speaker identification ({accuracy:.0%} accuracy)')

for i in range(len(speakers)):
    for j in range(len(speakers)):
        val = confusion[i, j]
        if val > 0:
            color = 'white' if val > confusion.max() / 2 else 'black'
            ax.text(j, i, str(val), ha='center', va='center',
                    color=color, fontsize=11, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Speaker feature space (t-SNE)

t-SNE projects the 12-dimensional MFCC space into 2D, preserving local structure. Each colour is a speaker. Clear clusters indicate the GMM has distinct speaker models to work with.

In [ ]:
# Subsample frames for t-SNE (too many points is slow and cluttered)
n_per_speaker = 200
X_all, y_all = [], []
for spk in speakers:
    data = train_features[spk]
    idx = np.random.choice(len(data), min(n_per_speaker, len(data)), replace=False)
    X_all.append(data[idx])
    y_all.extend([spk] * len(idx))

X_all = np.vstack(X_all)
y_all = np.array(y_all)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_2d = tsne.fit_transform(X_all)

fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.tab10
for i, spk in enumerate(speakers):
    mask = y_all == spk
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[cmap(i)], s=8, alpha=0.5, label=spk)

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('Speaker feature space (t-SNE of 12 MFCCs)')
ax.legend(markerscale=4, fontsize=8, ncol=2)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()